#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [8]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [9]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchain


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangGraph와 LangChain의 차이점

| 구분 | LangChain | LangGraph |
|------|-----------|-----------|
| **핵심 목표** | LLM(대형 언어 모델)을 활용한 **앱·시스템**을 빠르게 만들기 위한 **프레임워크** | LLM 기반 **워크플로우**를 **그래프(노드·엣지)** 형태로 설계·실행하기 위한 **라이브러리** |
| **주요 개념** | **Chain**, **Agent**, **Memory**, **PromptTemplate**, **Tool** 등 | **Node**, **Edge**, **Graph**, **State**(전역/노드‑별) |
| **워크플로우 모델링** | 순차적·조건부 체인(Chain)·에이전트(Agent) 중심. 복잡한 흐름을 구현하려면 직접 코드로 제어하거나 “router” 체인을 만들 필요가 있음. | **그래프** 형태로 흐름을 시각·코드로 정의. 노드 간 전이(Edge)를 명시하고, 상태(state)를 자동으로 전달·업데이트해 복잡한 비선형 흐름을 자연스럽게 표현. |
| **상태 관리** | 메모리(ConversationMemory, VectorStoreMemory 등)와 변수 전달을 수동으로 구현. | **State 객체**가 그래프 전체에 공유되고, 노드 실행 시 자동으로 읽고 쓰기. 상태 변화가 그래프 전이와 직접 연결돼 디버깅·테스트가 용이. |
| **실행 엔진** | `run()` 메서드가 체인을 순차적으로 실행. 에이전트는 LLM 호출 후 `tool` 선택 로직을 내부에 포함. | **GraphExecutor**가 그래프를 트래버스하면서 비동기·동기 실행을 지원. 각 노드는 독립적인 `Runnable`(LLM, 함수, API 등)일 수 있음. |
| **확장성** | 플러그인·툴(예: 검색, DB, 코드 실행) 추가가 용이하지만, 복합적인 흐름을 만들 때 구조가 복잡해질 수 있음. | 노드 단위로 **임의의 함수·